In [3]:
import pandas as pd
import re
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords

nltk.download('wordnet')
nltk.download('stopwords')
nltk.download('omw-1.4')

# Cargar datos
df = pd.read_csv('../../data/raw/youtoxic_english_1000.csv') 

# Recrear columna label
def categorize(row):
    if not row['IsToxic']:
        return 'clean'
    elif row['IsHatespeech'] or row['IsRacist']:
        return 'hate'
    elif row['IsAbusive']:
        return 'abusive'
    else:
        return 'toxic_other'

df['label'] = df.apply(categorize, axis=1)

# Preprocesamiento con nltk
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'\r\n|\r|\n', ' ', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = text.split()
    tokens = [
        lemmatizer.lemmatize(t) for t in tokens
        if t not in stop_words
        and len(t) > 2
    ]
    return ' '.join(tokens)

df['text_clean'] = df['Text'].apply(preprocess)
print(df[['Text', 'text_clean']].head(5))
print(f"\nComentarios vacíos: {(df['text_clean'] == '').sum()}")

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\zulay\AppData\Roaming\nltk_data...
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\zulay\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\zulay\AppData\Roaming\nltk_data...


                                                Text  \
0  If only people would just take a step back and...   
1  Law enforcement is not trained to shoot to app...   
2  \r\nDont you reckon them 'black lives matter' ...   
3  There are a very large number of people who do...   
4  The Arab dude is absolutely right, he should h...   

                                          text_clean  
0  people would take step back make case wasnt an...  
1  law enforcement trained shoot apprehend traine...  
2  dont reckon black life matter banner held whit...  
3  large number people like police officer called...  
4  arab dude absolutely right shot extra time sho...  

Comentarios vacíos: 0


In [5]:
df.to_csv('../../data/processed/comments_processed.csv', index=False)
print("Guardado OK")

Guardado OK
